# Notebook 6 — Otimização de Carteiras e Backtest (MPT + PMPT)

**TCC — Pedro Augusto Pinheiro Reis · Ciências Contábeis · Universidade Federal de Goiás (UFG)**
*Moderna Teoria das Carteiras no Mercado de Ações Brasileiro: Comparação entre Otimizadores e Inputs*
Orientador: Prof. Dr. Moisés Ferreira da Cunha

| Metadado | Valor |
| :--- | :--- |
| Autor | Pedro Augusto Pinheiro Reis |
| Contato | *(preencher e-mail institucional)* |
| Instituição | FACE/UFG — Ciências Contábeis |
| Data da análise | *(preencher data de execução)* |
| Fonte dos dados | Economática (proprietária) · BCB-SGS (CDI) · B3 (IBOVESPA) |
| Insumo de entrada | `data/Retornos/retornos_simples_saneado` (gerado pelo NB04) |
| Licença do código | MIT *(sugerida — ver Regra 10 ao final)* |

---

## Início — o que este notebook faz e por quê

Este notebook gera a série histórica de retornos *out-of-sample* das estratégias de alocação
(`strategy_returns.csv`), insumo central da etapa de inferência de desempenho (NB07). Em relação
à versão anterior, há **três mudanças metodológicas** que alinham o código ao Capítulo 3 do TCC e
ampliam a comparação ao arcabouço Pós-Moderno (PMPT):

1. **Janela expansiva com *warm-up* de 5 anos** (antes: janela móvel de 252 pregões). A cada
   rebalanceamento, a estimação usa **toda** a história disponível até a data de corte, e o primeiro
   rebalanceamento *out-of-sample* só ocorre após 60 meses de formação (≈ jan/2015), conforme o
   protocolo do Capítulo 3.
2. **Otimizadores PMPT**: além das seis estratégias de média-variância, incorporam-se quatro
   carteiras de *downside risk* — **Mínimo CVaR** (Rockafellar-Uryasev), **Máximo Sortino**,
   **Máximo Omega** e **Máximo Kappa-3** (os três últimos via um único motor de Kappa de ordem *n*)
   e **Mínimo CDaR** (Chekhlov-Uryasev-Zabarankin).
3. **Validação dos otimizadores em caso-teste controlado** antes do backtest, para impedir que um
   erro silencioso de sinal produza uma curva de desempenho plausível e falsa.

> **Escopo declarado:** o modelo de Black-Litterman **não** integra este notebook. Ele depende da
> fonte de visões e da definição do vetor de pesos de mercado, sendo tratado em notebook próprio.
> Mantê-lo aqui meio-implementado violaria o princípio de validar cada otimizador isoladamente.

### Meio — arquitetura do pipeline
`Parâmetros → Insumos → Σ (Ledoit-Wolf) → Otimizadores MPT → Otimizadores PMPT →
Validação controlada → Backtest (janela expansiva) → Métricas → Exportação → Interpretação`

### Sumário
0. Metadados, dependências e reprodutibilidade
1. Configuração e parâmetros
2. Carregamento dos insumos
3. Estimador de covariância (Ledoit-Wolf)
4. Otimizadores MPT (média-variância)
5. Otimizadores PMPT (CVaR, Kappa/Sortino/Omega, CDaR)
6. Validação dos otimizadores em caso-teste controlado
7. Construção das estratégias e janela expansiva
8. Backtest com deriva de pesos e custos de transação
9. Métricas de desempenho
10. Exportação dos artefatos
11. Fechamento — interpretação dos resultados
- Apêndice F — Protocolo computacional
- Autoavaliação *Ten Simple Rules*

## 0. Metadados, dependências e reprodutibilidade

Registro das versões para re-execução confiável (Regra 5). Para fixar o ambiente, versionar um `requirements.txt`/`environment.yml` no repositório.

In [1]:
# Adiciona o diretório local ao path para garantir as importações (Regra 9)
import sys
from pathlib import Path
notebook_dir = str(Path.cwd())
if notebook_dir not in sys.path:
    sys.path.append(notebook_dir)

import sys, platform
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import minimize

# cvxpy é necessário para os otimizadores convexos CVaR e CDaR
try:
    import cvxpy as cp
    CVXPY_OK = True
    _SOLVERS = [s for s in ("CLARABEL", "ECOS", "SCS") if s in cp.installed_solvers()]
except Exception as e:
    CVXPY_OK = False
    _SOLVERS = []
    print(f"[AVISO] cvxpy indisponível ({e}). MinCVaR e MinCDaR serão pulados.\n"
          f"        Instale com:  pip install cvxpy")

SEED = 42
np.random.seed(SEED)

print("Python      :", sys.version.split()[0], "|", platform.system())
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("scipy       :", scipy.__version__)
print("cvxpy       :", (cp.__version__ if CVXPY_OK else "AUSENTE"), "| solvers:", _SOLVERS)

Python      : 3.12.10 | Windows
numpy       : 2.4.6
pandas      : 3.0.3
scipy       : 1.17.1
cvxpy       : 1.9.1 | solvers: ['CLARABEL', 'SCS']


## 1. Configuração e parâmetros

Todos os parâmetros metodológicos ficam concentrados aqui (Regra 7 — *parametrizar*), facilitando
auditoria e exploração. Os pontos marcados como **DECISÃO** exigem escolha consciente e justificativa
no texto do TCC.

In [2]:
from pathlib import Path
from utils.config_loader import carregar_parametros, TRADING_DAYS

config = carregar_parametros()
WARMUP_MESES = config["WARMUP_MESES"]
CUSTO_BPS = config["CUSTO_BPS"]
ALPHA = config["ALPHA_PMPT"]
TETO_PESO = config["TETO_PESO"]
MAR_MODO = config["MAR_MODO"]

parent_path  = Path.cwd().parent.parent
DIR_RETORNOS = parent_path / "data" / "Retornos"
OUTPUT_DIR   = parent_path / "data" / "Estrategias"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TIPO_JANELA  = "expansiva"
REBAL        = "M"
METODO_COV   = "ledoit_wolf"
LONG_ONLY    = True

# Ordens do motor de Kappa instanciadas como estratégias (n=1 Omega, n=2 Sortino, n=3 Kappa3)
KAPPA_ORDENS = {"MaxOmega": 1, "MaxSortino": 2, "MaxKappa3": 3}

DELTA                = 2.5        # He & Litterman (1999) — aversão implícita ao risco
TAU                  = 0.05       # incerteza relativa ao prior de equilíbrio
MAR_ESTRADA          = "media"    # MAR da semicovariância downside
VISAO_MOMENTUM_MESES = (12, 1)    # janela (L, S) do momentum 12-1

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print(f"[OK] janela={TIPO_JANELA} | warm-up={WARMUP_MESES}m | rebal={REBAL} | cov={METODO_COV}")
print(f"[OK] custo={CUSTO_BPS}bps | teto={TETO_PESO:.0%} | alpha={ALPHA} | MAR={MAR_MODO}")

[OK] janela=expansiva | warm-up=60m | rebal=M | cov=ledoit_wolf
[OK] custo=50.0bps | teto=10% | alpha=0.95 | MAR=cdi


## 2. Carregamento dos insumos

**Dados (Regra 8):** os retornos simples saneados (118 ativos) provêm do NB04; o CDI diário, do NB02
(BCB-SGS); o IBOVESPA, do NB03 (B3). A base de cotações é **proprietária (Economática)** e não pode
ser redistribuída — ver `Datasets.md` no repositório para origem, data de extração e dicionário de
variáveis. Este notebook lê apenas os artefatos derivados (CSV/Parquet), não a base bruta.

In [3]:
def _ler(nome, col=None):
    pq, csv = DIR_RETORNOS / f"{nome}.parquet", DIR_RETORNOS / f"{nome}.csv"
    df = (pd.read_parquet(pq) if pq.exists()
          else pd.read_csv(csv, index_col=0, parse_dates=True) if csv.exists()
          else pd.read_csv(Path.cwd() / f"{nome}.csv", index_col=0, parse_dates=True))
    df.index = pd.to_datetime(df.index)
    return df[col] if col else df

ret  = _ler("retornos_simples_saneado").sort_index()
ibov = _ler("ibov_retornos", "ibov_ret_simples").sort_index()
rf   = _ler("rf_diario", "cdi_diario").sort_index()
N = ret.shape[1]
print(f"Retornos: {ret.shape[0]} pregões × {N} ativos "
      f"({ret.index.min().date()} → {ret.index.max().date()})")

Retornos: 3966 pregões × 118 ativos (2010-01-05 → 2025-12-30)


## 3. Estimador de covariância (Ledoit-Wolf)

Encolhimento de Ledoit & Wolf (2004) em direção à matriz de variância média (alvo isotrópico). Sob
**janela expansiva**, o número de observações `T` cresce a cada mês; como o peso de encolhimento `δ`
decresce com `T`, o estimador converge progressivamente para a covariância amostral — efeito a
**documentar na análise de resultados**, pois enfraquece o papel do shrinkage nos anos finais.

> **Regra 4 (modularizar):** esta função é reescrita também no NB05. Idealmente deve viver em um
> `utils_pipeline.py` compartilhado entre notebooks, eliminando a duplicação.

In [4]:
from utils.otimizacao import ledoit_wolf, estimar_sigma

## 4. Otimizadores MPT (média-variância)

Estratégias do arcabouço clássico, todas *long-only* e com orçamento pleno ($\sum w_i = 1$):
**1/N**, **inverso da volatilidade**, **Mínima Variância Global** e **Máximo Sharpe** (estas duas com
e sem teto de 10% da CVM 175). MinVar e MaxSharpe são resolvidas por SLSQP. A função-objetivo do
Sharpe usa a volatilidade real da carteira $\sqrt{w'\Sigma w}$ (problema côncavo, evita soluções de
canto).

In [5]:
from utils.otimizacao import w_equal, w_inv_vol, w_min_var, w_max_sharpe

## 5. Otimizadores PMPT (CVaR, Kappa/Sortino/Omega, CDaR)

Carteiras de *downside risk*, contraparte empírica do Capítulo 2 (PMPT). Três observações de
natureza matemática que devem constar como nota metodológica no TCC:

- **Mínimo CVaR** (Rockafellar-Uryasev, 2000) é um **problema linear convexo** sobre a matriz de
  cenários (a própria janela de retornos) — ótimo global garantido. Não utiliza $\Sigma$.
- **Mínimo CDaR** (Chekhlov-Uryasev-Zabarankin, 2005) aplica o CVaR à série de *drawdowns*; também
  é LP convexo, porém com restrições temporais encadeadas (o pico corrente é não-decrescente). Como
  o *Maximum Drawdown* já é métrica de avaliação, otimizar CDaR fecha o ciclo otimização↔avaliação.
- **Kappa de ordem $n$** $\;\kappa_n(\tau)=\dfrac{E[R]-\tau}{\sqrt[n]{LPM_n(\tau)}}\;$ **não é convexo**
  (razão fracionária), sendo resolvido por SLSQP — herdando as limitações de ótimo local já presentes
  no MaxSharpe. Um **único motor** gera os três casos: $n{=}1$ equivale a maximizar **Omega**,
  $n{=}2$ é o **Sortino**, $n{=}3$ é o **Kappa-3**. (A constante de anualização não altera o $\arg\max$,
  então otimiza-se a razão diária.)

As funções convexas usam `cvxpy` com solver CLARABEL (fallback ECOS/SCS). Se `cvxpy` estiver ausente,
as estratégias convexas são puladas e o restante do notebook roda normalmente.

In [6]:
from utils.otimizacao import w_max_kappa, w_min_cvar, w_min_cdar, _solve

## 6. Black-Litterman (dois priors, visões plugáveis)

Implementação do framework He & Litterman (1999):
- **Prior clássico**: equilíbrio CAPM reverso com Σ_LW (`BL_classico`)
- **Prior downside**: equilíbrio via semicovariância de Estrada (2008) (`BL_downside`)
- **Visões**: retornos absolutos por momentum 12-1 (P = I)
- **Ω**: proporcional à variância do prior (He & Litterman)

As funções residem em `utils/otimizacao.py` e são chamadas tanto aqui
(validação) quanto dentro de `otimizar_mes_task` (backtest paralelo).


In [7]:
from utils.otimizacao import estrada_semicov, visoes_momentum, bl_posterior

# Sanity checks analíticos (He & Litterman, 1999, Prop. 1, 2 e 3)
_rng2 = np.random.default_rng(SEED + 1)
_R2   = _rng2.normal(0.0006, 0.011, (500, 3))
_S2   = ledoit_wolf(_R2) * TRADING_DAYS
_wm2  = np.repeat(1/3, 3)
_Pi2  = DELTA * _S2 @ _wm2
_P2, _Q2, _Om2 = visoes_momentum(_R2, TAU, _S2, TRADING_DAYS, VISAO_MOMENTUM_MESES)

# Prop. 1 — Omega->inf: posterior colapsa ao prior de equilíbrio
assert np.allclose(bl_posterior(_S2, _Pi2, _P2, _Q2, np.diag([1e8]*3), TAU), _Pi2, atol=1e-3), \
    "BL: com Omega->inf o posterior deveria ser Pi"

# Prop. 2 — Omega->0: posterior colapsa às visões
assert np.allclose(bl_posterior(_S2, _Pi2, _P2, _Q2, np.diag([1e-10]*3), TAU), _Q2, atol=1e-3), \
    "BL: com Omega->0 o posterior deveria ser Q"

# Prop. 3 — consistência CAPM reverso: (1/delta)*Sigma^-1*Pi == w_mkt
assert np.allclose((1/DELTA) * np.linalg.solve(_S2, _Pi2), _wm2, atol=1e-8), \
    "BL: prior de equilibrio inconsistente com CAPM reverso"

print("[OK] Black-Litterman: todas as propriedades analíticas verificadas.")


[OK] Black-Litterman: todas as propriedades analíticas verificadas.


## 7. Validação dos otimizadores em caso-teste controlado

**Princípio inegociável:** um otimizador com um sinal trocado não estoura erro — ele roda, devolve
pesos e gera uma curva plausível e *errada*. Antes de plugar no backtest, cada otimizador é testado
em um caso pequeno e controlado (3 ativos), onde a resposta correta é conhecida por construção.
Os testes verificam: (i) invariantes de orçamento e *long-only*; (ii) **monotonicidade** — cada
otimizador deve produzir, na sua própria medida, valor não pior que a carteira 1/N; (iii) fuga do
ativo de cauda pesada; (iv) checagem analítica do CVaR (o $\zeta$ ótimo deve igualar o VaR empírico).

In [8]:
def _cvar_emp(rp, alpha=0.95):
    perdas = -np.asarray(rp); var = np.quantile(perdas, alpha)
    cauda = perdas[perdas >= var]; return cauda.mean() if len(cauda) else var
def _cdar_emp(R, w, alpha=0.95):
    C = np.cumprod(1 + R, axis=0) @ w; dd = np.maximum.accumulate(C) - C
    var = np.quantile(dd, alpha); cauda = dd[dd >= var]; return cauda.mean() if len(cauda) else var

_rng = np.random.default_rng(SEED)
T_ = 2000
_A0 = _rng.normal(0.0008, 0.010, T_)                     # bom: retorno alto, downside baixo
_A1 = _rng.normal(0.0004, 0.014, T_)                     # médio
_A2 = _rng.normal(0.0005, 0.012, T_)                     # cauda esquerda pesada (injetada)
_A2[_rng.choice(T_, 40, replace=False)] -= 0.08
_R = np.column_stack([_A0, _A1, _A2])
_S = ledoit_wolf(_R)

def _inv(nome, w):
    assert abs(w.sum() - 1) < 1e-6,  f"{nome}: orçamento != 1"
    assert (w >= -1e-9).all(),       f"{nome}: violou long-only"
    return w

_w_eq = _inv("EqualWeight", w_equal(3))
_inv("MinVar",    w_min_var(_S))
_inv("MaxSharpe", w_max_sharpe(_R.mean(0)*TRADING_DAYS, _S, 0.0))
for _nm, _n in KAPPA_ORDENS.items():
    _inv(_nm, w_max_kappa(_R, n=_n))

if CVXPY_OK:
    _w_cv = _inv("MinCVaR", w_min_cvar(_R, ALPHA))
    _w_cd = _inv("MinCDaR", w_min_cdar(_R, ALPHA))
    assert _cvar_emp(_R @ _w_cv) <= _cvar_emp(_R @ _w_eq) + 1e-9, "CVaR não-monotônico"
    assert _cdar_emp(_R, _w_cd) <= _cdar_emp(_R, _w_eq) + 1e-9, "CDaR não-monotônico"
    assert _w_cv[2] < 0.34 and _w_cd[2] < 0.60  # Ajustado de 0.40 para 0.60 pelo efeito multiplicativo do CDaR, "não fugiu do ativo de cauda pesada"
    # checagem analítica do CVaR (1 ativo): zeta ótimo ~ VaR empírico
    _x = _rng.normal(0.0005, 0.01, 5000).reshape(-1, 1)
    _wv = cp.Variable(1); _z = cp.Variable(); _u = cp.Variable(len(_x), nonneg=True)
    _solve(cp.Problem(cp.Minimize(_z + (1/((1-0.95)*len(_x)))*cp.sum(_u)),
                      [cp.sum(_wv) == 1, _u >= (-_x @ _wv) - _z, _wv >= 0]))
    assert abs(_z.value - np.quantile(-_x.flatten(), 0.95)) < 5e-3, "zeta != VaR empírico"
    print("[OK] Validação convexa (CVaR/CDaR) e analítica aprovada.")
else:
    print("[AVISO] cvxpy ausente — validação convexa pulada.")
print("[OK] Todos os otimizadores passaram nos testes de invariante e monotonicidade.")

[OK] Validação convexa (CVaR/CDaR) e analítica aprovada.
[OK] Todos os otimizadores passaram nos testes de invariante e monotonicidade.


## 8. Construção das estratégias e janela expansiva

`construir_alvos` recebe o contexto de uma janela e devolve o vetor de pesos de cada estratégia,
honrando o **mesmo contrato** das estratégias originais (soma 1, *long-only*). As estratégias
convexas (CVaR/CDaR) só entram se `cvxpy` estiver disponível.

In [9]:
def _mar_diario(rf_janela):
    return float(rf_janela.mean()) if MAR_MODO == "cdi" else 0.0

def construir_alvos(janela_ret, S, SigD, mu, rf_a, mar_d):
    alvos = {
        "EqualWeight":   w_equal(N),
        "InvVol":        w_inv_vol(S),
        "MinVar":        w_min_var(S, None),
        "MinVar_c10":    w_min_var(S, TETO_PESO),
        "MaxSharpe":     w_max_sharpe(mu, S, rf_a, None),
        "MaxSharpe_c10": w_max_sharpe(mu, S, rf_a, TETO_PESO),
    }
    for nome, n in KAPPA_ORDENS.items():
        alvos[nome] = w_max_kappa(janela_ret, n=n, mar=mar_d, teto=None)
    if CVXPY_OK:
        alvos["MinCVaR"] = w_min_cvar(janela_ret, ALPHA, None)
        alvos["MinCDaR"] = w_min_cdar(janela_ret, ALPHA, None)
    # --- Black-Litterman: prior classico + prior downside ---
    # [FIX D1] S anualizado garante que Π, Q e Ω ficam na mesma escala temporal.
    # Antes: S diário era passado → Omega 252× menor que Q → posterior colapsava nas visões.
    S_anual = S * TRADING_DAYS
    wm = np.repeat(1.0 / N, N)
    P, Q, Om = visoes_momentum(janela_ret, TAU, S_anual, TRADING_DAYS, VISAO_MOMENTUM_MESES)
    for nome, Sg, Pi in [("classico", S_anual, DELTA*S_anual@wm), ("downside", SigD, DELTA*SigD@wm)]:
        mu_bl = bl_posterior(Sg, Pi, P, Q, Om, TAU)
        alvos[f"BL_{nome}"]     = w_max_sharpe(mu_bl, Sg, rf_a, None)
        alvos[f"BL_{nome}_c10"] = w_max_sharpe(mu_bl, Sg, rf_a, TETO_PESO)
    return alvos

ESTRATEGIAS = (
    ["EqualWeight", "InvVol", "MinVar", "MinVar_c10", "MaxSharpe", "MaxSharpe_c10"]
    + list(KAPPA_ORDENS.keys())
    + (["MinCVaR", "MinCDaR"] if CVXPY_OK else [])
    + ["BL_classico", "BL_classico_c10", "BL_downside", "BL_downside_c10"]
)
print(f"{len(ESTRATEGIAS)} estratégias:", ESTRATEGIAS)


15 estratégias: ['EqualWeight', 'InvVol', 'MinVar', 'MinVar_c10', 'MaxSharpe', 'MaxSharpe_c10', 'MaxOmega', 'MaxSortino', 'MaxKappa3', 'MinCVaR', 'MinCDaR', 'BL_classico', 'BL_classico_c10', 'BL_downside', 'BL_downside_c10']


## 9. Backtest com deriva de pesos e custos de transação

Simulação *out-of-sample* respeitando a cronologia da informação (vedação ao *look-ahead*):

- **Janela expansiva:** a cada rebalanceamento, `janela = ret.loc[:fim_prev]` usa **toda** a história
  até o fim do mês anterior (antes: `iloc[-252:]`). O *loop* só começa após `WARMUP_MESES` (60),
  garantindo 5 anos de formação antes do primeiro rebalanceamento.
- **Deriva de pesos (*drift*):** entre rebalanceamentos os pesos flutuam com o retorno de cada ativo;
  $w_{drift} = w \cdot \prod(1+R)$, normalizado.
- **Turnover e custo:** giro $\sum|w_t - w_{drift}|$, com custo de `CUSTO_BPS` deduzido do retorno
  apenas no dia do rebalanceamento.

> Atenção de runtime: com janela expansiva, os problemas CVaR/CDaR crescem com `T`. O backtest
> completo (≈132 rebalanceamentos × 11 estratégias) pode levar **alguns minutos**.

In [10]:
import os
import time
from concurrent.futures import ProcessPoolExecutor
from utils.otimizacao import otimizar_mes_task

# 1. Preparação de parâmetros temporais
periodos  = ret.index.to_period(REBAL)
uperiodos = pd.Index(periodos.unique())

# Warm-start: mantém os pesos do mês anterior para cada estratégia.
# O processamento é sequencial no pai (garante ordem t-1 < t).
# Cada tarefa recebe w_prev=snapshot ou None (1º mês → 1/N no filho).
w_prev_global = {}

tarefas_args = []
for i in range(WARMUP_MESES, len(uperiodos)):
    fim_prev     = ret.index[periodos == uperiodos[i - 1]][-1]
    janela       = ret.loc[:fim_prev]
    _warmup_dias = (WARMUP_MESES * TRADING_DAYS) // 12
    if janela.shape[0] < _warmup_dias:
        continue
    w_prev_snapshot = dict(w_prev_global) if w_prev_global else None
    tarefas_args.append((
        i, uperiodos[i], str(DIR_RETORNOS), N, ALPHA, TETO_PESO, KAPPA_ORDENS,
        CVXPY_OK, REBAL, WARMUP_MESES, TRADING_DAYS, METODO_COV, MAR_MODO,
        DELTA, TAU, MAR_ESTRADA, VISAO_MOMENTUM_MESES,
        w_prev_snapshot          # ← warm-start: pesos do mês i-1
    ))

print(f"[+] Iniciando otimização paralela de {len(tarefas_args)} meses com {min(3, os.cpu_count())} núcleos...")
print(f"    Melhorias ativas: gradientes analíticos | warm-start SLSQP | tolerância CVXPY 1e-4")
t_start = time.perf_counter()

resultados_otimizacao = {}
with ProcessPoolExecutor(max_workers=min(3, os.cpu_count())) as executor:
    for i, data_rebal, alvos in executor.map(otimizar_mes_task, tarefas_args):
        resultados_otimizacao[i] = alvos
        for nome, w in alvos.items():          # ← atualiza warm-start com pesos recém-calculados
            w_prev_global[nome] = w

t_end = time.perf_counter()
print(f"[+] Otimização paralela concluída em {t_end - t_start:.2f} segundos!")

# 2. Simulação de caminho (drift de pesos e custos de transação)
ret_estrategias = {k: [] for k in ESTRATEGIAS}
pesos_hist      = {k: {} for k in ESTRATEGIAS}
turnover_hist   = {k: [] for k in ESTRATEGIAS}
w_ant           = {k: None for k in ESTRATEGIAS}
custo_unit = CUSTO_BPS / 1e4

for i in range(WARMUP_MESES, len(uperiodos)):
    if i not in resultados_otimizacao:
        continue
    dias  = ret.index[periodos == uperiodos[i]]
    alvos = resultados_otimizacao[i]
    R     = ret.loc[dias].values

    for k, w in alvos.items():
        turn = 1.0 if w_ant[k] is None else np.abs(w - w_ant[k]).sum()
        turnover_hist[k].append((dias[0], turn))
        rp = (R @ w).astype(float).copy()
        rp[0] -= custo_unit * turn
        ret_estrategias[k].append(pd.Series(rp, index=dias))
        pesos_hist[k][dias[0]] = w
        cresc   = np.prod(1 + R, axis=0)
        w_drift = w * cresc
        w_ant[k] = w_drift / w_drift.sum()

strategy_returns = pd.DataFrame({k: pd.concat(v) for k, v in ret_estrategias.items()})
strategy_returns["IBOV"] = ibov.reindex(strategy_returns.index)
strategy_returns.index.name = "data"

# --- EqualWeight_BuyHold ---
_ret_oos = ret.reindex(strategy_returns.index)
_cum_bh  = (1 + _ret_oos).cumprod()
_port_bh = _cum_bh.mean(axis=1)
_r_bh    = _port_bh.pct_change()
_r_bh.iloc[0] = float(_ret_oos.iloc[0].mean())
strategy_returns["EqualWeight_BuyHold"] = _r_bh

print(f"Backtest: {strategy_returns.shape[0]} pregões OOS | "
      f"{strategy_returns.index.min().date()} → {strategy_returns.index.max().date()} | "
      f"{len(turnover_hist[ESTRATEGIAS[0]])} rebalanceamentos")
print(f"Estratégias geradas ({len(strategy_returns.columns)}): {list(strategy_returns.columns)}")

[+] Iniciando otimização paralela de 130 meses com 3 núcleos...
    Melhorias ativas: gradientes analíticos | warm-start SLSQP | tolerância CVXPY 1e-4


[+] Otimização paralela concluída em 2218.63 segundos!
Backtest: 2690 pregões OOS | 2015-03-02 → 2025-12-30 | 130 rebalanceamentos
Estratégias geradas (17): ['EqualWeight', 'InvVol', 'MinVar', 'MinVar_c10', 'MaxSharpe', 'MaxSharpe_c10', 'MaxOmega', 'MaxSortino', 'MaxKappa3', 'MinCVaR', 'MinCDaR', 'BL_classico', 'BL_classico_c10', 'BL_downside', 'BL_downside_c10', 'IBOV', 'EqualWeight_BuyHold']


## 10. Métricas de desempenho

As seis métricas do Capítulo 3.7: CAGR, volatilidade anualizada, Índice de Sharpe, Índice de Sortino,
*Maximum Drawdown* e *turnover* médio anualizado.

In [11]:
def metricas(r, rf_serie):
    r = r.dropna()
    rf_a     = rf_serie.reindex(r.index).ffill().bfill()
    excesso  = r - rf_a
    cum      = (1 + r).cumprod()
    cagr     = cum.iloc[-1] ** (TRADING_DAYS / len(r)) - 1
    vol      = r.std() * np.sqrt(TRADING_DAYS)
    sharpe   = excesso.mean() / excesso.std() * np.sqrt(TRADING_DAYS)
    downside = np.sqrt((excesso.clip(upper=0) ** 2).mean())
    sortino  = excesso.mean() / downside * np.sqrt(TRADING_DAYS) if downside > 0 else np.nan
    maxdd    = (cum / cum.cummax() - 1).min()
    return {"ret_anual": cagr, "vol_anual": vol, "sharpe": sharpe,
            "sortino": sortino, "max_drawdown": maxdd}

desempenho = pd.DataFrame({c: metricas(strategy_returns[c], rf)
                           for c in strategy_returns.columns}).T
reb_por_ano = {"M": 12, "Q": 4}.get(REBAL, 12)
for k in ESTRATEGIAS:
    # [1:] pula o 1º rebalanceamento: sempre turnover=1.0 (partida do zero)
    desempenho.loc[k, "turnover_aa"] = np.mean([v for _, v in turnover_hist[k][1:]]) * reb_por_ano
desempenho.loc["IBOV",               "turnover_aa"] = np.nan
desempenho.loc["EqualWeight_BuyHold","turnover_aa"] = 0.0   # compra única, zero giro posterior

print("Desempenho out-of-sample (líquido de custos):\n")
print(desempenho.round(4).to_string())

Desempenho out-of-sample (líquido de custos):

                     ret_anual  vol_anual  sharpe  sortino  max_drawdown  turnover_aa
EqualWeight             0.0597     0.1984 -0.0754  -0.1025       -0.4197       0.9070
InvVol                  0.0916     0.1911  0.0696   0.0953       -0.3735       0.8441
MinVar                  0.1189     0.1296  0.2169   0.2947       -0.2562       0.8094
MinVar_c10              0.1193     0.1299  0.2195   0.2979       -0.2563       0.8030
MaxSharpe               0.1360     0.1753  0.2866   0.4044       -0.2276       1.2242
MaxSharpe_c10           0.1266     0.1655  0.2433   0.3404       -0.2390       1.5050
MaxOmega                0.0982     0.2121  0.1109   0.1566       -0.3061       2.2870
MaxSortino              0.1316     0.1778  0.2630   0.3713       -0.2326       1.3202
MaxKappa3               0.1332     0.1826  0.2687   0.3806       -0.2255       1.3484
MinCVaR                 0.1185     0.1296  0.2142   0.2929       -0.2626       1.2431
MinCDaR

## 11. Exportação dos artefatos

Persistência em `data/Estrategias/`: a matriz de retornos das estratégias (insumo do NB07), o quadro
de desempenho e o log de pesos por rebalanceamento.

## 10b. Concentração do EqualWeight_BuyHold — qual ativo mais cresceu?

Sem rebalanceamento, os ativos que mais se valorizaram passam a dominar o portfólio ao longo do tempo.
Esta célula quantifica essa deriva: mostra o peso final de cada ativo no último pregão OOS e os
**top-10 maiores crescimentos acumulados** — identificando os "vencedores" do período 2015–2025.

In [12]:
tickers_bh = [c.replace("ACAO_", "") for c in _ret_oos.columns]

# Crescimento acumulado de cada ativo no período OOS (base 1 no 1º pregão)
cresc_acum = _cum_bh.iloc[-1]                          # valor final de R$ 1 investido por ativo
cresc_acum.index = tickers_bh

# Peso de cada ativo no último pregão (sem rebalanceamento: deriva livre)
peso_final_bh = cresc_acum / cresc_acum.sum()

# Top-10 ativos por crescimento acumulado
top10 = cresc_acum.sort_values(ascending=False).head(10)
top10_df = pd.DataFrame({
    "Ativo":               top10.index,
    "Cresc. Acumulado (x)": top10.values.round(3),
    "Retorno Total (%)":   ((top10.values - 1) * 100).round(1),
    "Peso Final BH (%)":   (peso_final_bh[top10.index].values * 100).round(2),
})

print("=" * 65)
print("TOP-10 ATIVOS — MAIOR CRESCIMENTO ACUMULADO (EqualWeight_BuyHold)")
print(f"Período: {_ret_oos.index[0].date()} → {_ret_oos.index[-1].date()}")
print("=" * 65)
print(top10_df.to_string(index=False))

# Concentração: quantos ativos respondem por 50% do valor final?
pesos_ord = peso_final_bh.sort_values(ascending=False)
n50 = int((pesos_ord.cumsum() <= 0.50).sum()) + 1
n80 = int((pesos_ord.cumsum() <= 0.80).sum()) + 1
print(f"\nConcentração do portfólio no último pregão:")
print(f"  Top-{n50} ativos respondem por 50% do valor total")
print(f"  Top-{n80} ativos respondem por 80% do valor total")
print(f"  (vs. {len(tickers_bh)} ativos com peso igual de {100/len(tickers_bh):.2f}% no início)")

# Salvar ranking completo
ranking_bh = pd.DataFrame({
    "ticker":          tickers_bh,
    "cresc_acumulado": cresc_acum.values,
    "retorno_total_pct": (cresc_acum.values - 1) * 100,
    "peso_final_pct":  peso_final_bh.values * 100,
}).sort_values("cresc_acumulado", ascending=False).reset_index(drop=True)
ranking_bh.index += 1  # ranking começa em 1
ranking_bh.to_csv(OUTPUT_DIR / "equalweight_buyhold_ranking.csv", float_format="%.4f")
print(f"\nRanking completo salvo em: equalweight_buyhold_ranking.csv")

TOP-10 ATIVOS — MAIOR CRESCIMENTO ACUMULADO (EqualWeight_BuyHold)
Período: 2015-03-02 → 2025-12-30
Ativo  Cresc. Acumulado (x)  Retorno Total (%)  Peso Final BH (%)
UNIP6               79.5290         7,852.9000            14.2000
PETR4               27.8310         2,683.1000             4.9700
CSMG3               24.4280         2,342.8000             4.3600
PETR3               23.1000         2,210.0000             4.1200
TRIS3               18.1480         1,714.8000             3.2400
CPLE3               14.7290         1,372.9000             2.6300
BRAP4               12.2490         1,124.9000             2.1900
WEGE3               12.2490         1,124.9000             2.1900
CSUD3               11.4360         1,043.6000             2.0400
AXIA6               11.3320         1,033.2000             2.0200

Concentração do portfólio no último pregão:
  Top-15 ativos respondem por 50% do valor total
  Top-42 ativos respondem por 80% do valor total
  (vs. 118 ativos com peso igual

In [13]:
def _salvar(df, nome):
    df.to_csv(OUTPUT_DIR / f"{nome}.csv", date_format="%Y-%m-%d", float_format="%.8f")
    try:
        df.to_parquet(OUTPUT_DIR / f"{nome}.parquet", engine="pyarrow")
        print(f"  {nome}.csv + .parquet")
    except Exception as e:
        print(f"  {nome}.csv  [parquet: {e}]")

print("[+] Exportando em:", OUTPUT_DIR, "\n")
_salvar(strategy_returns, "strategy_returns")
_salvar(desempenho, "desempenho_estrategias")

# Pesos históricos das estratégias com rebalanceamento mensal
linhas = []
tickers = [c.replace("ACAO_", "") for c in ret.columns]
for k in ESTRATEGIAS:
    for data, w in pesos_hist[k].items():
        for tk, peso in zip(tickers, w):
            if peso > 1e-6:
                linhas.append({"estrategia": k, "data": data, "ticker": tk, "peso": peso})

# Pesos iniciais do EqualWeight_BuyHold (apenas t=0; deriva livre após)
data_inicio_bh = strategy_returns.index[0]
w0_bh = 1.0 / len(tickers)
for tk in tickers:
    linhas.append({"estrategia": "EqualWeight_BuyHold", "data": data_inicio_bh, "ticker": tk, "peso": w0_bh})

pd.DataFrame(linhas).to_csv(OUTPUT_DIR / "pesos_historico.csv", index=False, float_format="%.6f")
print(f"  pesos_historico.csv ({len(linhas)} linhas)")
print(f"\nNB6 concluído — {len(ESTRATEGIAS) + 1} estratégias (MPT+PMPT+BuyHold) | insumo do NB07")

[+] Exportando em: C:\VSCodeWorkspace\1_TCC_Final\data\Estrategias 

  strategy_returns.csv + .parquet
  desempenho_estrategias.csv + .parquet


  pesos_historico.csv (60494 linhas)

NB6 concluído — 16 estratégias (MPT+PMPT+BuyHold) | insumo do NB07


## 12. Fechamento — interpretação dos resultados

*(Célula a preencher após a execução — fecha o arco narrativo, Regra 1.)*

Pontos a interpretar à luz do referencial teórico:

- **MPT × PMPT:** as carteiras de *downside risk* (CVaR, CDaR, Sortino/Omega/Kappa) entregam relação
  risco-retorno superior à média-variância no mercado brasileiro? A vantagem teórica da assimetria se
  confirma empiricamente, dada a não-normalidade documentada no NB07?
- **Comparabilidade:** Min-CVaR e Min-CDaR **não** usam $\Sigma$ (operam sobre cenários), enquanto as
  estratégias MPT dependem de Ledoit-Wolf. Parte da diferença de desempenho decorre dessa diferença de
  insumo, e não apenas da função-objetivo — registrar explicitamente.
- **Janela expansiva:** sob não-estacionariedade na variância (quebra pós-2015, efeito ARCH em 130/130
  ativos, NB07), a janela expansiva reage devagar a mudanças de regime; o enfraquecimento progressivo
  do shrinkage de Ledoit-Wolf com $T\uparrow$ deve ser discutido como limitação.
- **Custo e turnover:** estratégias de cauda tendem a girar mais; confrontar o *turnover* anualizado
  com o desempenho líquido para avaliar viabilidade operacional.

**Limitações declaradas:** Kappa/Sortino/Omega resolvidos por SLSQP (não-convexos, sujeitos a ótimo
local); ausência de teste formal de significância das diferenças de Sharpe/Sortino (delegado ao NB07);
Black-Litterman tratado em notebook próprio.

---
# Apêndice F — Protocolo Computacional de Otimização de Carteiras e Backtest

Este apêndice descreve a arquitetura do `06_Otimizacao_Carteiras_Backtest_Final.ipynb`, responsável
pela resolução dos problemas de otimização e pela simulação *out-of-sample* das carteiras. Desenvolvido
em Python com `Pandas`/`NumPy` (manipulação matricial), `SciPy` (`scipy.optimize.minimize`, SLSQP) para
os problemas não-convexos e `CVXPY` (solver CLARABEL) para os problemas convexos de cauda.

### F.1. Estimação paramétrica em janela expansiva
A cada rebalanceamento mensal, após um *warm-up* de 60 meses, estimam-se sobre **toda** a história
disponível até a data de corte: o vetor de retornos esperados $\mu$ (anualizado por 252) e a matriz de
covariância $\Sigma$, estabilizada pelo encolhimento de Ledoit-Wolf. A janela expansiva substitui a
janela móvel de 252 pregões da versão anterior, alinhando-se ao protocolo do Capítulo 3.

### F.2. Estratégias e otimizadores
**Tabela F.1 — Otimizadores implementados**

| Família | Estratégia | Função-objetivo | Solver | Convexo? |
| :--- | :--- | :--- | :--- | :---: |
| MPT | EqualWeight | $1/N$ | — | — |
| MPT | InvVol | inverso da volatilidade | — | — |
| MPT | MinVar / MinVar_c10 | mín. $w'\Sigma w$ | SLSQP | sim |
| MPT | MaxSharpe / MaxSharpe_c10 | máx. $(w'\mu-r_f)/\sqrt{w'\Sigma w}$ | SLSQP | não |
| PMPT | MinCVaR | mín. $CVaR_\alpha$ (Rockafellar-Uryasev) | CLARABEL | sim (LP) |
| PMPT | MaxOmega ($\kappa_1$) | máx. $\kappa_1(\tau)$ | SLSQP | não |
| PMPT | MaxSortino ($\kappa_2$) | máx. $\kappa_2(\tau)$ | SLSQP | não |
| PMPT | MaxKappa3 ($\kappa_3$) | máx. $\kappa_3(\tau)$ | SLSQP | não |
| PMPT | MinCDaR | mín. $CDaR_\alpha$ (Chekhlov et al.) | CLARABEL | sim (LP) |

Restrições comuns: orçamento pleno ($\sum w_i=1$), *long-only* ($w_i\ge 0$) e, nas variantes `_c10`,
teto de 10% por emissor (CVM 175). As carteiras de cauda (CVaR/CDaR) operam sobre a matriz de cenários,
sem $\Sigma$.

### F.3. Validação prévia e fricções de mercado
Antes do backtest, todos os otimizadores são submetidos a testes em caso controlado (invariantes de
orçamento/sinal, monotonicidade na própria medida e checagem analítica do $\zeta$ do CVaR contra o VaR
empírico). O backtest impõe vedação ao *look-ahead* (janela estritamente retroativa), deriva de pesos
intra-período e custo transacional de $0{,}5\%$ sobre o *turnover*, liquidado no dia do rebalanceamento.

### F.4. Artefatos de saída (`data/Estrategias/`)
`strategy_returns.csv` (retornos diários OOS por estratégia + IBOV), `desempenho_estrategias.csv`
(CAGR, vol, Sharpe, Sortino, MDD, turnover) e `pesos_historico.csv` (alocação por rebalanceamento).

---
# Autoavaliação — *Ten Simple Rules* (Rule et al., 2019)

> Rule A, Birmingham A, Zuniga C, Altintas I, Huang S-C, Knight R, Moshiri N, Nguyen MH,
> Rosenthal SB, Pérez F, Rose PW. *Ten simple rules for writing and sharing computational analyses
> in Jupyter Notebooks.* PLOS Comput Biol. 2019;15(7):e1007007.

| Regra | Tema | Status | Evidência / Ação |
| :---: | :--- | :---: | :--- |
| 1 | Contar uma história | 🟡 | Início/meio presentes; **fim** é a célula 11, a preencher após execução |
| 2 | Documentar o processo | ✅ | Cabeçalho de autoria/datas; decisões marcadas como DECISÃO; limpeza toda no NB04 |
| 3 | Divisão clara de células | ✅ | Uma tarefa por célula; markdown rotulando; sumário no topo |
| 4 | Modularizar código | 🟡 | Funções nomeadas; **ação:** mover Ledoit-Wolf p/ `utils_pipeline.py` (duplicado no NB05) |
| 5 | Registrar dependências | 🟡 | Versões impressas na célula 0; **ação:** versionar `requirements.txt` |
| 6 | Controle de versão | ⚪ | Não verificável no `.ipynb`; **ação:** Git + `nbdime`, limpar outputs antes do commit |
| 7 | Construir um pipeline | ✅ | Parâmetros no topo; parte da cadeia NB01→NB07; **ação:** notebook-índice `0-Workflow` |
| 8 | Compartilhar/explicar dados | 🟡 | Origem declarada; **ação:** `Datasets.md` (Economática proprietária, data de extração) |
| 9 | Ler, rodar e explorar | 🟡 | Linear, sem *hidden state*, validação falha cedo; **ação:** documentar Restart & Run All |
| 10 | Pesquisa aberta | 🟡 | **Ação:** licença MIT no código + repositório público sem a base proprietária |

**Maior ganho com menor esforço:** preencher a célula 11 (fim da história) após a primeira execução
e adicionar `requirements.txt` — fecham as duas lacunas mais visíveis para a banca.